# Mapeo de códigos

## Inicializa Spark

In [0]:
%run "/Workspace/Project/etl_censo_databricks/notebooks/00_Init_Spark"

## Lee tablas de códigos

In [0]:
df_ct = spark.read.table("bronze.codigos_territorios")

In [0]:
df_cte = spark.read.table("bronze.codigos_territorios_especificos")

In [0]:
df_co = spark.read.table("bronze.codigos_otros")

## Función para el cruce entre tablas de entrada y Códigos Territoriales

In [0]:
def cruce_codigos_territoriales(df):
  df_cruce_reg = df.alias("v").join(
    broadcast(df_ct.alias("ct")), 
    [col("v.region") == col("ct.codigo_territorial"), col("ct.division_politica") == "Región"],
    "inner")

  # Elimina campos originales y resultados del cruce que no son necesarios
  df_cruce_reg = df_cruce_reg.drop("region", "division_politica", "codigo_territorial")
  # Renombra el campo 'territorio'
  df_cruce_reg = df_cruce_reg.withColumnRenamed("territorio", "region")

  df_cruce_prov = df_cruce_reg.alias("cr").join(
      broadcast(df_ct.alias("ct")), 
      [col("cr.provincia") == col("ct.codigo_territorial"), col("ct.division_politica") == "Provincia"], 
      "inner")

  # Elimina campos originales y resultados del cruce que no son necesarios
  df_cruce_prov = df_cruce_prov.drop("provincia", "division_politica", "codigo_territorial")
  # Renombra el campo 'territorio'
  df_cruce_prov = df_cruce_prov.withColumnRenamed("territorio", "provincia")

  df_cruce = df_cruce_prov.alias("cp").join(
      broadcast(df_ct.alias("ct")), 
      [col("cp.comuna") == col("ct.codigo_territorial"), col("ct.division_politica") == "Comuna"], 
      "inner")

  # Elimina campos originales y resultados del cruce que no son necesarios
  df_cruce = df_cruce.drop("comuna", "division_politica", "codigo_territorial")
  # Renombra el campo 'territorio'
  df_cruce = df_cruce.withColumnRenamed("territorio", "comuna")

  return df_cruce


## Función para el cruce entre tablas de entrada y Códigos Varios

In [0]:
def cruce_codigos_otros(df, cols_gb):
    df_long = df.select(*cols_gb,
        explode(
            array([
                struct(lit(c).alias("campo"), col(c).cast("string").alias("codigo")) 
                for c in df.columns
            ])
        ).alias("kv")
    ).select(*cols_gb, col("kv.campo").alias("co_campo"), col("kv.codigo"))


    joined = df_long.join(
        broadcast(df_co),
        on=[df_long.co_campo == df_co.campo, df_long.codigo == df_co.codigo],
        how="left"
    ) .withColumn("descripcion", \
            when(df_co.descripcion.isNull(), df_long.codigo) \
            # No respuesta. Dejo null.
            .when(df_long.codigo == "-99", lit(None)) \
            # Valor suprimido por anonimizacion. Dejo null.
            .when(df_long.codigo == "-66", lit(None))
            .otherwise(df_co.descripcion)) \
            .select(*cols_gb, df_long.co_campo, "descripcion")

    df_desc = joined \
            .groupBy(*cols_gb) \
            .pivot("co_campo") \
            .agg(first("descripcion"))

    # Filtro los campos
    # Si existen en df_desc tomo ese, si no existe tomo el del DF original
    # ya que df_desc tiene los valores de los codigos
    cols_final = [col(f"df_desc.{c}") for c in df.columns if c in df_desc.columns] + [col(f"df.{c}") for c in df.columns if c not in df_desc.columns]

    # Recupero todos los campos 
    df_final = df.alias("df").join(df_desc.alias("df_desc"), on=cols_gb, how="inner").select(*cols_final)

    return df_final

## Cruza tablas de entrada con Códigos Territoriales

### Cruce entre Vivienda y Códigos Territoriales

In [0]:
df_vivienda = spark.read.table("bronze.viviendas")

In [0]:
df_vivienda_ct = cruce_codigos_territoriales(df_vivienda)

### Cruce entre Hogar y Códigos Territoriales

In [0]:
df_hogar = spark.read.table("bronze.hogares")

In [0]:
df_hogar_ct = cruce_codigos_territoriales(df_hogar)

### Cruce entre Persona y Códigos Territoriales

In [0]:
df_persona = spark.read.table("bronze.personas")

In [0]:
df_persona_ct = cruce_codigos_territoriales(df_persona)

### Cruce entre Persona y Códigos Territoriales específicos

In [0]:
# Campos de codigos territoriales especificos
campos_cte = [
    "p25_lug_nacimiento_esp",
    "p27_nacionalidad_esp",
    "p44_lug_trab_esp",
    "p24_lug_resid5_esp"]

for c_cte in campos_cte:
    df_persona_ct = df_persona_ct \
        .alias("df_pct") \
        .join(broadcast(df_cte.alias(f"df_cte")), col(f"df_pct.{c_cte}") == col(f"df_cte.codigo_especifico"), "left") \
        .withColumn(c_cte, 
                    when(col("df_cte.codigo_especifico") > 0, col("df_cte.territorio_especifico")) \
                    .when(col(f"df_pct.{c_cte}") == "-66", lit(None)) \
                    .when(col(f"df_pct.{c_cte}") == "-99", lit(None))
                    .otherwise(c_cte)) \
        .drop("codigo_especifico", "territorio_especifico")


## Cruce tablas de entrada con Códigos Varios

### Cruce entre Vivienda y Códigos Varios

In [0]:
df_vivienda_co = cruce_codigos_otros(df_vivienda_ct, ["id_vivienda"])
display(df_vivienda_co)

### Cruce entre Hogar y Códigos Varios

In [0]:
df_hogar_co = cruce_codigos_otros(df_hogar_ct, ["id_vivienda", "id_hogar"])
display(df_hogar_co)

### Cruce entre Persona y Códigos Varios

In [0]:
df_persona_co = cruce_codigos_otros(df_persona_ct, ["id_vivienda", "id_hogar", "id_persona"])
display(df_persona_co)

## Guarda el resultado en las tablas

In [0]:
df_vivienda_co.write.insertInto("silver.viviendas", overwrite=True)

In [0]:
df_hogar_co.write.insertInto("silver.hogares", overwrite=True)

In [0]:

df_persona_co.write.insertInto("silver.personas", overwrite=True)